> **Privacidad y reproducción.** Esta copia no incluye fuentes clínicas, nóminas ni datos fila a fila. Antes de ejecutarla, configure `MDS_NOSHOW_ROOT` como la carpeta del proyecto que contiene `02_Datos_Fuente` y `MDS_NOSHOW_HMAC_KEY` con una clave aleatoria privada de al menos 32 bytes. La clave no debe guardarse en GitHub. Los identificadores internos se seudonimizan con HMAC-SHA256; los datos de origen permanecen fuera del repositorio.

# No-show en APS — CESFAM El Monte

Notebook analítico, reproducible y auditable para construir una línea base operacional, examinar el proceso de contacto y preparar una valorización financiera responsable.

**Regla principal:** ninguna falta de vínculo se interpreta automáticamente como no-show y ningún gasto agregado se interpreta automáticamente como pérdida de caja.

## tl;dr

- La línea base construida arroja una **tasa candidata de no-show de 11,42%**: 16.543 no-shows sobre 144.800 citas candidatas, después de excluir 629 claves que aparecen simultáneamente como atención cerrada e inasistencia.
- Esta cifra **no es todavía la tasa institucional validada**: el reporte de actividades no distingue explícitamente agenda programada elegible, atención espontánea y cancelaciones en un universo común.
- Keirón se vincula con **28,5%** de las citas candidatas mediante nombre exacto unívoco y día. La cobertura por RUT es insuficiente (319 registros en Keirón), por lo que el vínculo nominal sigue siendo exploratorio.
- Las confirmadas presentan menor no-show candidato que “sin responder” o “falla de envío”; las canceladas muestran una inconsistencia elevada con Rayen. Son asociaciones y problemas de conciliación, no evidencia causal del recordatorio.
- El balance de gastos reconcilia $4.287.010.964 devengados a junio de 2026. La nómina de julio reconcilia $471.982.808 en haberes y 323,93 FTE equivalentes.
- La dotación vincula 96,6% de la nómina y permite valorizar 90,5% de los no-shows. Bajo el escenario central de **45 minutos**, **70% de jornada clínica** y gasto total absorbido, la **pérdida económica estimada por capacidad clínica financiada no utilizada es $293.634.370 entre enero y julio**, equivalente a $41.947.767 mensuales.
- Decisión provisional: **mantener el problema** y comunicar el monto como pérdida económica operacional. Validar duración, proporción clínica, unidades, demanda y reasignación antes de atribuir una pérdida evitable a una intervención.


## Contexto, decisiones y preguntas

La decisión es si el problema de inasistencia mantiene prioridad institucional, debe reformularse por segmento o requiere primero mejorar la evidencia. El análisis responde:

1. ¿Cuál es la magnitud observada y la tasa candidata de no-show?
2. ¿Dónde se concentra por mes, instrumento, tipo de atención y sector?
3. ¿Qué cobertura y resultados muestra el contacto de Keirón?
4. ¿Qué parte de la información permite valorar capacidad financiada no utilizada?
5. ¿Qué bloqueos impiden declarar una tasa institucional o un costo por cita?

### Supuestos clave

- Período operacional principal: enero–julio de 2026.
- Período financiero del balance: enero–junio de 2026.
- Nómina: julio de 2026 solamente.
- Los reportes de actividades e inasistencias no comparten un identificador explícito de cita; se construyen claves candidatas y se cuantifican ambigüedades.
- Las salidas nunca muestran nombres, RUT, teléfonos, direcciones, previsión ni remuneraciones individuales.

In [ ]:
from pathlib import Path
from html.parser import HTMLParser
from IPython.display import display, Markdown
from openpyxl import load_workbook
import hashlib, hmac, io, os, re, subprocess, tempfile, unicodedata, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 30)
plt.style.use('seaborn-v0_8-whitegrid')

ROOT = Path(os.environ.get('MDS_NOSHOW_ROOT', Path.cwd())).expanduser()
HMAC_KEY = os.environ.get('MDS_NOSHOW_HMAC_KEY', '')
if len(HMAC_KEY.encode('utf-8')) < 32:
    raise RuntimeError('Configure MDS_NOSHOW_HMAC_KEY con al menos 32 bytes aleatorios; nunca la guarde en el repositorio.')
PERIODO_INICIO = pd.Timestamp('2026-01-01')
PERIODO_FIN_OPERACIONAL = pd.Timestamp('2026-07-31 23:59:59')
PERIODO_FIN_FINANCIERO = pd.Timestamp('2026-06-30 23:59:59')
MIN_GRUPO_PUBLICABLE = 10
SEMANAS_PROMEDIO_MES = 4.33
MOSTRAR_PII = False

ARCHIVOS = {
    'actividades_2026': ROOT / '02_Datos_Fuente/01_Atenciones_e_Inasistencias/Monitoreo_de_Actividades (16).xls',
    'inasistentes_2026': ROOT / '02_Datos_Fuente/01_Atenciones_e_Inasistencias/Monitoreo_de_Inasistentes (2).xls',
    'inasistentes_julio': ROOT / '02_Datos_Fuente/01_Atenciones_e_Inasistencias/inasistentes julio .xlsx',
    'keiron': ROOT / '02_Datos_Fuente/01_Atenciones_e_Inasistencias/negocios_cesfam_el_monte_2026_08_19_2121.csv',
    'balance_detalle': ROOT / '02_Datos_Fuente/02_Finanzas_y_Dotacion/balance salud maximo detalle 2026.xlsx',
    'nomina_julio': ROOT / '02_Datos_Fuente/01_Atenciones_e_Inasistencias/Informe Salud.xls',
    'dotacion_agosto': ROOT / '02_Datos_Fuente/02_Finanzas_y_Dotacion/PLANILLA DOTACION AGOSTO 2026.xlsx',
}

faltantes = [str(p.name) for p in ARCHIVOS.values() if not p.exists()]
assert not faltantes, f'Faltan archivos requeridos: {faltantes}'
print('Parámetros cargados. PII visible:', MOSTRAR_PII)

## Fuentes, períodos y trazabilidad

Se calcula hash de cada archivo para que otra persona pueda verificar que trabaja con la misma versión. Los hashes no contienen información personal.

In [ ]:
def sha256_archivo(path, chunk=1024*1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for bloque in iter(lambda: f.read(chunk), b''):
            h.update(bloque)
    return h.hexdigest()

inventario = pd.DataFrame([
    {'fuente': nombre, 'archivo': p.name, 'bytes': p.stat().st_size, 'sha256': sha256_archivo(p)}
    for nombre, p in ARCHIVOS.items()
])
display(inventario)

## Carga segura y funciones auxiliares

Los archivos con extensión `.xls` de Rayen son HTML. Se procesan por flujo para evitar cargar millones de celdas innecesarias. Sólo se retienen columnas operacionales y los identificadores se transforman inmediatamente en hashes.

In [ ]:
def texto_norm(valor):
    if pd.isna(valor): return ''
    s = unicodedata.normalize('NFKD', str(valor))
    s = ''.join(c for c in s if not unicodedata.combining(c))
    return re.sub(r'[^A-Z0-9]+', ' ', s.upper()).strip()

def rut_norm(valor):
    return re.sub(r'[^0-9K]', '', texto_norm(valor))

def hmac_id(valor, prefijo):
    if pd.isna(valor): return np.nan
    limpio = str(valor).strip()
    if not limpio: return np.nan
    mensaje = f'{prefijo}|{limpio}'.encode('utf-8')
    return hmac.new(HMAC_KEY.encode('utf-8'), mensaje, hashlib.sha256).hexdigest()[:32]

def hash_id(valor, prefijo):
    limpio = rut_norm(valor)
    if not limpio: return np.nan
    return hmac_id(limpio, prefijo)

class TablaHTMLStream(HTMLParser):
    def __init__(self, tabla_objetivo, columnas):
        super().__init__(convert_charrefs=True)
        self.tabla_objetivo = tabla_objetivo
        self.columnas = columnas
        self.tabla = -1
        self.en_tabla = False
        self.en_fila = False
        self.en_celda = False
        self.tipo_celda = None
        self.buffer = []
        self.fila = []
        self.encabezado = None
        self.indices = None
        self.registros = []
    def handle_starttag(self, tag, attrs):
        tag = tag.lower()
        if tag == 'table':
            self.tabla += 1
            self.en_tabla = self.tabla == self.tabla_objetivo
        elif self.en_tabla and tag == 'tr':
            self.en_fila = True; self.fila = []
        elif self.en_fila and tag in ('td','th'):
            self.en_celda = True; self.tipo_celda = tag; self.buffer = []
    def handle_data(self, data):
        if self.en_celda: self.buffer.append(data)
    def handle_endtag(self, tag):
        tag = tag.lower()
        if self.en_celda and tag in ('td','th'):
            self.fila.append(' '.join(''.join(self.buffer).split()))
            self.en_celda = False
        elif self.en_fila and tag == 'tr':
            if self.fila:
                if self.encabezado is None and any(texto_norm(x) in {texto_norm(c) for c in self.columnas} for x in self.fila):
                    self.encabezado = self.fila
                    mapa = {texto_norm(v): i for i,v in enumerate(self.encabezado)}
                    self.indices = {c: mapa.get(texto_norm(c)) for c in self.columnas}
                elif self.indices:
                    rec = {c: (self.fila[i] if i is not None and i < len(self.fila) else '') for c,i in self.indices.items()}
                    self.registros.append(rec)
            self.en_fila = False
        elif tag == 'table' and self.en_tabla:
            self.en_tabla = False

def leer_tabla_html(path, columnas, tabla=1):
    parser = TablaHTMLStream(tabla, columnas)
    with open(path, 'r', encoding='utf-8', errors='replace') as f:
        for bloque in iter(lambda: f.read(1024*1024), ''):
            parser.feed(bloque)
    parser.close()
    return pd.DataFrame(parser.registros)

def fecha_segura(serie):
    return pd.to_datetime(serie, dayfirst=True, errors='coerce')

def suprimir_grupos(df, col_n='n'):
    out = df.copy()
    mascara = out[col_n] < MIN_GRUPO_PUBLICABLE
    for c in out.columns:
        if c != col_n and pd.api.types.is_numeric_dtype(out[c]): out.loc[mascara, c] = np.nan
    out.loc[mascara, col_n] = np.nan
    return out

## Datos Rayen: actividades e inasistencias

La unidad de cita no viene explícita. Se construye una clave candidata con paciente seudonimizado, fecha, funcionario, instrumento y tipo de atención. La clave se conserva como hipótesis técnica, no como identificador institucional validado.

In [ ]:
cols_act = ['N°','ESTADO ATENCIÓN','FECHA CONSULTA','DURACIÓN','FECHA CITA','TIPO DE ATENCIÓN','SECTOR CITA','FUNCIONARIO','INSTRUMENTO ','RUN','PACIENTE']
act_raw = leer_tabla_html(ARCHIVOS['actividades_2026'], cols_act)
act_raw = act_raw.rename(columns={'N°':'nro','ESTADO ATENCIÓN':'estado','FECHA CONSULTA':'fecha_consulta','DURACIÓN':'duracion','FECHA CITA':'fecha_cita','TIPO DE ATENCIÓN':'tipo_atencion','SECTOR CITA':'sector','FUNCIONARIO':'funcionario','INSTRUMENTO ':'instrumento','RUN':'rut','PACIENTE':'paciente'})
act_raw['fecha_cita'] = fecha_segura(act_raw['fecha_cita'])
act_raw['fecha_consulta'] = fecha_segura(act_raw['fecha_consulta'])
act_raw['duracion'] = pd.to_numeric(act_raw['duracion'], errors='coerce')
act_raw['rut_hash'] = act_raw['rut'].map(lambda x: hash_id(x, 'paciente'))
act_raw['nombre_hash'] = act_raw['paciente'].map(lambda x: hmac_id(texto_norm(x), 'nombre') if texto_norm(x) else np.nan)

# Filas primarias: estado y fecha presentes. Las filas de procedimientos secundarios se auditan por separado.
filas_primarias = act_raw['estado'].str.strip().ne('') & act_raw['fecha_cita'].notna() & act_raw['rut_hash'].notna()
act = act_raw.loc[filas_primarias, ['nro','estado','fecha_consulta','duracion','fecha_cita','tipo_atencion','sector','funcionario','instrumento','rut_hash','nombre_hash']].copy()
act = act[(act['fecha_cita'] >= PERIODO_INICIO) & (act['fecha_cita'] <= PERIODO_FIN_OPERACIONAL)]

cols_ina = ['FUNCIONARIO','INSTRUMENTO','FECHA CITA','TIPO ATENCION','RUN','SECTOR','USUARIO']
ina_raw = leer_tabla_html(ARCHIVOS['inasistentes_2026'], cols_ina)
ina_raw = ina_raw.rename(columns={'FUNCIONARIO':'funcionario','INSTRUMENTO':'instrumento','FECHA CITA':'fecha_cita','TIPO ATENCION':'tipo_atencion','RUN':'rut','SECTOR':'sector','USUARIO':'paciente'})
ina_raw['fecha_cita'] = fecha_segura(ina_raw['fecha_cita'])
ina_raw['rut_hash'] = ina_raw['rut'].map(lambda x: hash_id(x, 'paciente'))
ina_raw['nombre_hash'] = ina_raw['paciente'].map(lambda x: hmac_id(texto_norm(x), 'nombre') if texto_norm(x) else np.nan)
ina = ina_raw.loc[ina_raw['fecha_cita'].notna() & ina_raw['rut_hash'].notna(), ['funcionario','instrumento','fecha_cita','tipo_atencion','sector','rut_hash','nombre_hash']].copy()
ina = ina[(ina['fecha_cita'] >= PERIODO_INICIO) & (ina['fecha_cita'] <= PERIODO_FIN_OPERACIONAL)]

for df in (act, ina):
    for c in ['funcionario','instrumento','tipo_atencion','sector']:
        df[c+'_norm'] = df[c].map(texto_norm)
    df['fecha_dia'] = df['fecha_cita'].dt.normalize()
    df['clave_candidata'] = (
        df['rut_hash'].astype(str) + '|' + df['fecha_dia'].astype(str) + '|' +
        df['funcionario_norm'] + '|' + df['instrumento_norm'] + '|' + df['tipo_atencion_norm']
    ).map(lambda x: hmac_id(x, 'cita'))

perfil_rayen = pd.DataFrame([
    {'fuente':'Actividades: filas HTML', 'n':len(act_raw)},
    {'fuente':'Actividades: filas primarias en período', 'n':len(act)},
    {'fuente':'Actividades: claves candidatas únicas', 'n':act['clave_candidata'].nunique()},
    {'fuente':'Inasistencias: filas HTML', 'n':len(ina_raw)},
    {'fuente':'Inasistencias: filas utilizables en período', 'n':len(ina)},
    {'fuente':'Inasistencias: claves candidatas únicas', 'n':ina['clave_candidata'].nunique()},
])
display(perfil_rayen)

## Auditoría de calidad y construcción de la cita candidata

Se cuantifican duplicados y solapamientos. Si una misma clave aparece como cerrada e inasistente, se clasifica como ambigua y se excluye de la tasa primaria.

In [ ]:
act_u = act.sort_values('fecha_consulta').drop_duplicates('clave_candidata', keep='last').copy()
ina_u = ina.drop_duplicates('clave_candidata').copy()
solapadas = set(act_u['clave_candidata']) & set(ina_u['clave_candidata'])

calidad_citas = pd.DataFrame([
    {'prueba':'Duplicados en actividades según clave candidata', 'afectados':len(act)-len(act_u), 'severidad':'Alta'},
    {'prueba':'Duplicados en inasistencias según clave candidata', 'afectados':len(ina)-len(ina_u), 'severidad':'Media'},
    {'prueba':'Claves presentes como cerrada e inasistente', 'afectados':len(solapadas), 'severidad':'Crítica'},
    {'prueba':'Filas secundarias de actividades sin fecha/estado', 'afectados':int((~filas_primarias).sum()), 'severidad':'Informativa'},
    {'prueba':'Inasistencia sin fecha interpretable', 'afectados':int(ina_raw['fecha_cita'].isna().sum()), 'severidad':'Alta'},
])
display(calidad_citas)

act_valid = act_u[~act_u['clave_candidata'].isin(solapadas)].copy()
ina_valid = ina_u[~ina_u['clave_candidata'].isin(solapadas)].copy()
act_valid['resultado'] = 'atencion_cerrada'
ina_valid['resultado'] = 'no_show'
citas_candidatas = pd.concat([
    act_valid[['clave_candidata','rut_hash','nombre_hash','fecha_dia','fecha_cita','instrumento_norm','tipo_atencion_norm','sector_norm','resultado']],
    ina_valid[['clave_candidata','rut_hash','nombre_hash','fecha_dia','fecha_cita','instrumento_norm','tipo_atencion_norm','sector_norm','resultado']],
], ignore_index=True)

assert citas_candidatas['clave_candidata'].is_unique
assert not citas_candidatas['rut_hash'].isna().any()
print('Citas candidatas reconciliadas:', len(citas_candidatas))

## Línea base operacional candidata

La fórmula es `no-show / (atenciones cerradas + no-show)` después de excluir ambigüedades. **No es todavía la tasa institucional validada**, porque el reporte de actividades puede incluir atenciones no programadas y no registra cancelaciones elegibles en el mismo universo.

In [ ]:
def intervalo_wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k/n; den = 1+z*z/n
    centro = (p+z*z/(2*n))/den
    margen = z*np.sqrt((p*(1-p)+z*z/(4*n))/n)/den
    return centro-margen, centro+margen

n_total = len(citas_candidatas)
n_no_show = int((citas_candidatas['resultado']=='no_show').sum())
tasa_candidata = n_no_show/n_total if n_total else np.nan
ic_inf, ic_sup = intervalo_wilson(n_no_show, n_total)

linea_base = pd.DataFrame([{
    'denominador_candidato':n_total,
    'no_show':n_no_show,
    'tasa_candidata':tasa_candidata,
    'ic95_inf':ic_inf,
    'ic95_sup':ic_sup,
    'estado_validacion':'NO VALIDADA: falta distinguir agenda programada elegible y cancelaciones'
}])
display(linea_base.style.format({'tasa_candidata':'{:.2%}','ic95_inf':'{:.2%}','ic95_sup':'{:.2%}'}))

## Segmentación y concentración

Se muestran volumen y tasa juntos. Los grupos con menos de 10 citas se suprimen.

In [ ]:
citas_candidatas['mes'] = citas_candidatas['fecha_dia'].dt.to_period('M').astype(str)
citas_candidatas['es_no_show'] = (citas_candidatas['resultado']=='no_show').astype(int)

def resumen_segmento(col, top=20):
    x = citas_candidatas.groupby(col, dropna=False).agg(n=('es_no_show','size'), no_show=('es_no_show','sum'), tasa=('es_no_show','mean')).reset_index()
    x = x.sort_values(['no_show','n'], ascending=False).head(top)
    return suprimir_grupos(x)

mensual = resumen_segmento('mes', top=20)
instrumentos = resumen_segmento('instrumento_norm', top=20)
display(mensual.style.format({'tasa':'{:.1%}'}))
display(instrumentos.style.format({'tasa':'{:.1%}'}))

fig, ax = plt.subplots(figsize=(9,4.5))
plot_m = mensual.dropna(subset=['n'])
ax.bar(plot_m['mes'], plot_m['tasa']*100, color='#2C6E8F')
ax.set(title='Tasa candidata de no-show por mes', ylabel='Porcentaje', xlabel='Mes')
ax.tick_params(axis='x', rotation=35)
plt.tight_layout(); plt.show()

## Comprobación del archivo de julio

El archivo mensual debe ser subconjunto del reporte enero–julio. Se compara mediante una clave candidata equivalente; no se suma al acumulado.

In [ ]:
julio_raw = pd.read_excel(ARCHIVOS['inasistentes_julio'], header=8, dtype=str)
julio_raw.columns = [str(c).strip() for c in julio_raw.columns]
julio = julio_raw.rename(columns={'FUNCIONARIO':'funcionario','INSTRUMENTO':'instrumento','FECHA CITA':'fecha_cita','TIPO ATENCION':'tipo_atencion','RUN':'rut','SECTOR':'sector'}).copy()
julio['fecha_cita'] = fecha_segura(julio['fecha_cita'])
julio['rut_hash'] = julio['rut'].map(lambda x: hash_id(x, 'paciente'))
for c in ['funcionario','instrumento','tipo_atencion','sector']:
    julio[c+'_norm'] = julio[c].map(texto_norm)
julio['fecha_dia'] = julio['fecha_cita'].dt.normalize()
julio['clave_candidata'] = (
    julio['rut_hash'].astype(str) + '|' + julio['fecha_dia'].astype(str) + '|' + julio['funcionario_norm'] + '|' + julio['instrumento_norm'] + '|' + julio['tipo_atencion_norm']
).map(lambda x: hmac_id(x, 'cita'))
julio_keys = set(julio.loc[julio['rut_hash'].notna() & julio['fecha_cita'].notna(),'clave_candidata'])
acum_keys = set(ina_u['clave_candidata'])
display(pd.DataFrame([{
    'claves_julio':len(julio_keys),
    'contenidas_en_acumulado':len(julio_keys & acum_keys),
    'cobertura':len(julio_keys & acum_keys)/len(julio_keys) if julio_keys else np.nan,
    'decision':'No sumar; usar sólo como control de cobertura'
}]).style.format({'cobertura':'{:.2%}'}))

## Keirón: embudo de contacto y vinculación con Rayen

Se usa primero RUT seudonimizado y día de cita. Como Keirón sólo contiene RUT en una fracción pequeña, se aplica en segundo lugar nombre exacto normalizado y día, únicamente cuando ese nombre corresponde a un solo RUT en Rayen. Los nombres nunca aparecen en salidas.

In [ ]:
keiron = pd.read_csv(ARCHIVOS['keiron'], dtype=str, low_memory=False)
keiron.columns = [c.lstrip('﻿').strip() for c in keiron.columns]
keiron['fecha_cita'] = fecha_segura(keiron['Fecha y hora cita'])
keiron['fecha_actualizacion'] = fecha_segura(keiron['Fecha de actualización'])
keiron['rut_hash'] = keiron['Rut'].map(lambda x: hash_id(x, 'paciente'))
keiron['nombre_hash'] = keiron['Nombre del paciente'].map(lambda x: hmac_id(texto_norm(x), 'nombre') if texto_norm(x) else np.nan)
keiron['fecha_dia'] = keiron['fecha_cita'].dt.normalize()
keiron['estado_norm'] = keiron['Estado'].map(texto_norm)

perfil_keiron = pd.DataFrame([
    {'metrica':'Registros', 'valor':len(keiron)},
    {'metrica':'Con fecha de cita interpretable', 'valor':int(keiron['fecha_cita'].notna().sum())},
    {'metrica':'Con RUT utilizable', 'valor':int(keiron['rut_hash'].notna().sum())},
    {'metrica':'Con nombre utilizable (sólo hash interno)', 'valor':int(keiron['nombre_hash'].notna().sum())},
    {'metrica':'Nombre-día con múltiples estados', 'valor':int(keiron.groupby(['nombre_hash','fecha_dia'],dropna=True)['estado_norm'].nunique().gt(1).sum())},
])
display(perfil_keiron)

embudo = keiron['estado_norm'].replace('', 'SIN ESTADO').value_counts(dropna=False).rename_axis('estado').reset_index(name='n')
embudo['porcentaje'] = embudo['n']/len(keiron)
display(embudo.style.format({'porcentaje':'{:.1%}'}))

k_rut = keiron[keiron['rut_hash'].notna() & keiron['fecha_dia'].notna()].sort_values('fecha_actualizacion').drop_duplicates(['rut_hash','fecha_dia'], keep='last')
k_nombre = keiron[keiron['nombre_hash'].notna() & keiron['fecha_dia'].notna()].sort_values('fecha_actualizacion').drop_duplicates(['nombre_hash','fecha_dia'], keep='last')

# Un nombre sólo se admite si mapea a un único RUT en todo Rayen.
mapa_nombre = citas_candidatas.dropna(subset=['nombre_hash','rut_hash']).groupby('nombre_hash')['rut_hash'].nunique()
nombres_univocos = set(mapa_nombre[mapa_nombre==1].index)
k_nombre = k_nombre[k_nombre['nombre_hash'].isin(nombres_univocos)]

cruce = citas_candidatas.merge(k_rut[['rut_hash','fecha_dia','estado_norm']], on=['rut_hash','fecha_dia'], how='left', validate='many_to_one')
cruce = cruce.rename(columns={'estado_norm':'estado_rut'})
cruce = cruce.merge(k_nombre[['nombre_hash','fecha_dia','estado_norm']], on=['nombre_hash','fecha_dia'], how='left', validate='many_to_one')
cruce = cruce.rename(columns={'estado_norm':'estado_nombre'})
conflictos_metodo = int((cruce['estado_rut'].notna() & cruce['estado_nombre'].notna() & (cruce['estado_rut']!=cruce['estado_nombre'])).sum())
cruce['estado_norm'] = cruce['estado_rut'].fillna(cruce['estado_nombre'])
cruce['metodo_match'] = np.select([cruce['estado_rut'].notna(), cruce['estado_nombre'].notna()], ['RUT + día','Nombre unívoco + día'], default='Sin vínculo')
cobertura_keiron = cruce['estado_norm'].notna().mean()
estado_resultado = cruce[cruce['estado_norm'].notna()].groupby('estado_norm').agg(n=('es_no_show','size'), no_show=('es_no_show','sum'), tasa_candidata=('es_no_show','mean')).reset_index().sort_values('n',ascending=False)
display(pd.DataFrame([{'citas_candidatas':len(cruce),'vinculadas_keiron':int(cruce['estado_norm'].notna().sum()),'por_rut':int((cruce['metodo_match']=='RUT + día').sum()),'por_nombre_univoco':int((cruce['metodo_match']=='Nombre unívoco + día').sum()),'conflictos_entre_metodos':conflictos_metodo,'cobertura':cobertura_keiron}]).style.format({'cobertura':'{:.1%}'}))
display(suprimir_grupos(estado_resultado).style.format({'tasa_candidata':'{:.1%}'}))

## Estructura financiera: balance detallado

La exportación mezcla separadores regionales y una columna porcentual defectuosa. Se normalizan montos, se recalculan porcentajes y se usan cuentas ancla para QA. No se suman padres e hijos.

In [ ]:
def monto_balance(v):
    if v is None or v == '': return 0.0
    if isinstance(v, str):
        return float(v.replace(',','').strip())
    v = float(v)
    if v != 0 and abs(v) < 10000 and not v.is_integer():
        return round(v*1000)
    return v

wsb = load_workbook(ARCHIVOS['balance_detalle'], data_only=True).active
reg_balance = []
for r in range(9, wsb.max_row+1):
    codigo = wsb.cell(r,1).value
    if not codigo: continue
    reg_balance.append({
        'fila':r, 'codigo':str(codigo), 'denominacion':str(wsb.cell(r,2).value or '').strip(),
        'presupuesto_vigente':monto_balance(wsb.cell(r,4).value),
        'obligado_acum':monto_balance(wsb.cell(r,7).value),
        'devengado_acum':monto_balance(wsb.cell(r,10).value),
        'pagado_acum':monto_balance(wsb.cell(r,12).value),
    })
balance = pd.DataFrame(reg_balance)
balance['pct_obligado_recalculado'] = np.where(balance['presupuesto_vigente']!=0, balance['obligado_acum']/balance['presupuesto_vigente'], np.nan)
balance['nivel_cuenta'] = balance['codigo'].str.split('-').map(lambda xs: next((i for i,s in enumerate(xs[1:],1) if int(s)==0), len(xs)-1)-1)

anclas_codigos = {
    '215-00-00-000-000-000': 4287010964,
    '215-21-00-000-000-000': 3513497147,
    '215-22-00-000-000-000': 393541368,
    '215-22-04-004-001-002': 97206923,
    '215-22-04-005-001-001': 43055307,
    '215-22-11-999-001-001': 80980130,
}
qa_balance=[]
for cod, esperado in anclas_codigos.items():
    observado=float(balance.loc[balance['codigo']==cod,'devengado_acum'].iloc[0])
    qa_balance.append({'codigo':cod,'esperado':esperado,'observado':observado,'diferencia':observado-esperado,'resultado':'PASS' if observado==esperado else 'FAIL'})
display(pd.DataFrame(qa_balance))

cuentas_relevantes = balance[balance['codigo'].isin(anclas_codigos)].copy()
display(cuentas_relevantes[['codigo','denominacion','devengado_acum','pagado_acum','pct_obligado_recalculado']].style.format({'devengado_acum':'${:,.0f}','pagado_acum':'${:,.0f}','pct_obligado_recalculado':'{:.1%}'}))

## Nómina de julio: reconstrucción privada y salida agregada

El `.xls` se convierte temporalmente sin modificar el original. No se retienen nombres, RUT, salud previsional ni descuentos individuales. El costo disponible es **haber bruto**, no costo total empleador.

In [ ]:
SOFFICE = Path(os.environ.get('SOFFICE_BIN', 'soffice'))
with tempfile.TemporaryDirectory(prefix='nomina_no_show_') as td:
    perfil = Path(td)/'perfil_lo'
    cmd = [str(SOFFICE), f'-env:UserInstallation=file://{perfil}', '--headless', '--convert-to', 'xlsx', '--outdir', td, str(ARCHIVOS['nomina_julio'])]
    proc = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
    convertido = Path(td)/'Informe Salud.xlsx'
    if not convertido.exists():
        raise RuntimeError('No fue posible convertir la nómina: '+proc.stderr[-500:])
    wsn = load_workbook(convertido, data_only=True).active
    nomina=[]
    for r in range(25, 2824):
        codigo = wsn.cell(r,1).value
        categoria = wsn.cell(r+1,2).value if r+1 <= wsn.max_row else None
        if isinstance(codigo,(int,float)) and categoria in list('ABCDEF'):
            nomina.append({
                'funcionario_hash':hmac_id(codigo, 'funcionario'),
                'categoria':categoria,
                'nivel':wsn.cell(r+1,3).value,
                'dias_trabajados':wsn.cell(r+1,4).value or 0,
                'jornada_semanal':wsn.cell(r+1,5).value or 0,
                'total_imponible':wsn.cell(r+1,7).value or 0,
                'total_tributable':wsn.cell(r+1,9).value or 0,
                'total_haberes':wsn.cell(r+2,11).value or 0,
            })
    nomina = pd.DataFrame(nomina)
    totales_pie = {
        'total_imponible':wsn['B2840'].value,
        'total_tributable':wsn['B2842'].value,
        'total_haberes':wsn['B2853'].value,
    }

qa_nomina = pd.DataFrame([
    {'prueba':'327 registros', 'observado':len(nomina), 'esperado':327},
    {'prueba':'Total imponible', 'observado':nomina['total_imponible'].sum(), 'esperado':totales_pie['total_imponible']},
    {'prueba':'Total tributable', 'observado':nomina['total_tributable'].sum(), 'esperado':totales_pie['total_tributable']},
    {'prueba':'Total haberes', 'observado':nomina['total_haberes'].sum(), 'esperado':totales_pie['total_haberes']},
])
qa_nomina['resultado'] = np.where(qa_nomina['observado']==qa_nomina['esperado'],'PASS','FAIL')
display(qa_nomina)
assert (qa_nomina['resultado']=='PASS').all()
assert nomina['funcionario_hash'].is_unique

nomina_cat = nomina.groupby('categoria').agg(n=('funcionario_hash','size'), horas_semanales=('jornada_semanal','sum'), total_haberes=('total_haberes','sum')).reset_index()
nomina_cat['fte_44h'] = nomina_cat['horas_semanales']/44
nomina_cat['haber_por_hora_contratada_aprox'] = nomina_cat['total_haberes']/(nomina_cat['horas_semanales']*SEMANAS_PROMEDIO_MES)
nomina_publica = suprimir_grupos(nomina_cat)
display(nomina_publica.style.format({'total_haberes':'${:,.0f}','fte_44h':'{:,.2f}','haber_por_hora_contratada_aprox':'${:,.0f}'}))
display(pd.DataFrame([{
    'registros_dias_cero':int((nomina['dias_trabajados']==0).sum()),
    'haberes_dias_cero':nomina.loc[nomina['dias_trabajados']==0,'total_haberes'].sum(),
    'estado':'PENDIENTE DE INTERPRETACIÓN POR REMUNERACIONES'
}]))


## Dotación de agosto y primera aproximación de costo de capacidad

La dotación permite asociar la nómina agregada con cargo/profesión y jornada mediante un identificador interno. La nómina corresponde a julio y la dotación a agosto; por ello el cruce se usa como aproximación y no como costeo contable definitivo.

La métrica monetaria es **capacidad salarial valorizada asociada a no-show**. No representa ahorro de caja ni pérdida incremental: el personal continúa contratado. Para traducirla a valor recuperable se aplican escenarios de duración del cupo, factor de costo empleador y proporción efectivamente reasignada.


In [ ]:

# Carga privada: nunca se muestran nombre ni RUT.
dot_raw = pd.read_excel(ARCHIVOS['dotacion_agosto'], header=2, dtype=str)
dot = dot_raw.iloc[:, :8].copy()
dot.columns = ['establecimiento','id_interno','rut','nombre','cargo','jornada_dotacion','tipo_contrato','nivel_dotacion']
dot['id_interno'] = pd.to_numeric(dot['id_interno'], errors='coerce')
dot = dot[dot['id_interno'].notna()].copy()
dot['id_interno'] = dot['id_interno'].astype(int)
dot['jornada_dotacion'] = pd.to_numeric(dot['jornada_dotacion'], errors='coerce')
dot['funcionario_hash'] = dot['id_interno'].map(lambda x: hmac_id(x, 'funcionario'))
dot['cargo_norm'] = dot['cargo'].map(texto_norm)
dot['establecimiento_norm'] = dot['establecimiento'].map(texto_norm)
dot['contrato_norm'] = dot['tipo_contrato'].map(texto_norm)

# Unificaciones semánticas mínimas y visibles; no se fuerzan equivalencias ambiguas.
dot['cargo_analitico'] = dot['cargo_norm'].replace({
    'MATRON': 'MATRONAS',
    'AUXILIAR DENTAL': 'TECNICO DENTAL',
})

qa_dotacion = pd.DataFrame([
    {'prueba':'Registros de dotación', 'observado':len(dot), 'resultado':'PASS' if len(dot)==318 else 'REVISAR'},
    {'prueba':'ID interno único', 'observado':dot['id_interno'].nunique(), 'resultado':'PASS' if dot['id_interno'].is_unique else 'FAIL'},
    {'prueba':'Cargo completo', 'observado':int(dot['cargo_norm'].ne('').sum()), 'resultado':'PASS' if dot['cargo_norm'].ne('').all() else 'FAIL'},
    {'prueba':'Jornada completa', 'observado':int(dot['jornada_dotacion'].notna().sum()), 'resultado':'PASS' if dot['jornada_dotacion'].notna().all() else 'FAIL'},
    {'prueba':'FTE equivalentes 44 h', 'observado':dot['jornada_dotacion'].sum()/44, 'resultado':'INFORMATIVO'},
])
display(qa_dotacion)

nomina_dotacion = nomina.merge(
    dot[['funcionario_hash','cargo_analitico','establecimiento_norm','jornada_dotacion','contrato_norm']],
    on='funcionario_hash', how='left', validate='one_to_one', indicator=True
)
cobertura_personas = nomina_dotacion['cargo_analitico'].notna().mean()
cobertura_haberes = nomina_dotacion.loc[nomina_dotacion['cargo_analitico'].notna(),'total_haberes'].sum()/nomina_dotacion['total_haberes'].sum()
coincidencia_jornada = np.isclose(
    nomina_dotacion.loc[nomina_dotacion['jornada_dotacion'].notna(),'jornada_semanal'].astype(float),
    nomina_dotacion.loc[nomina_dotacion['jornada_dotacion'].notna(),'jornada_dotacion'].astype(float),
).mean()

qa_cruce_dotacion = pd.DataFrame([{
    'nomina_julio':len(nomina_dotacion),
    'vinculados_dotacion':int(nomina_dotacion['cargo_analitico'].notna().sum()),
    'cobertura_personas':cobertura_personas,
    'cobertura_haberes':cobertura_haberes,
    'coincidencia_jornada_julio_agosto':coincidencia_jornada,
    'advertencia':'Períodos distintos: nómina julio / dotación agosto'
}])
display(qa_cruce_dotacion.style.format({
    'cobertura_personas':'{:.1%}', 'cobertura_haberes':'{:.1%}',
    'coincidencia_jornada_julio_agosto':'{:.1%}'
}))

# Haberes por hora contratada. Excluye aportes patronales y no equivale a hora clínica.
costo_cargo = (
    nomina_dotacion[nomina_dotacion['cargo_analitico'].notna()]
    .groupby('cargo_analitico')
    .agg(funcionarios=('funcionario_hash','size'), total_haberes=('total_haberes','sum'), horas_semanales=('jornada_semanal','sum'))
    .reset_index()
)
costo_cargo['fte_44h'] = costo_cargo['horas_semanales']/44
costo_cargo['haber_hora_contratada'] = costo_cargo['total_haberes']/(costo_cargo['horas_semanales']*SEMANAS_PROMEDIO_MES)

# Puente explícito entre instrumentos Rayen y cargos de dotación.
MAPA_INSTRUMENTO_CARGO = {
    'MEDICO':'MEDICOS',
    'ODONTOLOGO A':'ODONTOLOGOS',
    'ENFERMERO A':'ENFERMERAS',
    'KINESIOLOGO A':'KINESIOLOGO',
    'NUTRICIONISTA':'NUTRICIONISTA',
    'MATRON A':'MATRONAS',
    'PSICOLOGO A':'PSICOLOGO',
    'TERAPEUTA OCUPACIONAL':'TERAPEUTA OCUPACIONAL',
    'TRABAJADOR A SOCIAL':'ASISTENTE SOCIAL',
    'FONOAUDIOLOGO A':'FONOAUDIOLOGO',
    'TECNICO PARAMEDICO DENTAL':'TECNICO DENTAL',
    'PROFESOR A DE EDUCACION FISICA':'PROFESORA EDUCACION FISICA',
    'QUIMICO FARMACEUTICO':'QUIMICO FARMACEUTICO',
    'EDUCADOR A DE PARVULOS':'EDUCADORA DE PARVULOS',
}

no_show_instrumento = (
    citas_candidatas[citas_candidatas['resultado']=='no_show']
    .groupby('instrumento_norm').size().rename('no_show').reset_index()
)
no_show_instrumento['cargo_analitico'] = no_show_instrumento['instrumento_norm'].map(MAPA_INSTRUMENTO_CARGO)
costeo_profesion = no_show_instrumento.merge(
    costo_cargo[['cargo_analitico','funcionarios','haber_hora_contratada']],
    on='cargo_analitico', how='left', validate='many_to_one'
)
no_show_total = int(no_show_instrumento['no_show'].sum())
no_show_mapeado = int(costeo_profesion.loc[costeo_profesion['haber_hora_contratada'].notna(),'no_show'].sum())
cobertura_no_show_costeable = no_show_mapeado/no_show_total

# Escenario central: 30 minutos y haberes (factor empleador 1,0).
DURACION_CENTRAL_MIN = 30
costeo_profesion['costo_capacidad_central'] = (
    costeo_profesion['no_show'] * DURACION_CENTRAL_MIN/60 * costeo_profesion['haber_hora_contratada']
)
costeo_publico = costeo_profesion[costeo_profesion['haber_hora_contratada'].notna()].copy()
costeo_publico = costeo_publico.sort_values('costo_capacidad_central', ascending=False)
display(costeo_publico[['instrumento_norm','no_show','funcionarios','haber_hora_contratada','costo_capacidad_central']].style.format({
    'haber_hora_contratada':'${:,.0f}', 'costo_capacidad_central':'${:,.0f}'
}))

# Sensibilidad: duración y factor para aproximar costo total empleador.
escenarios=[]
for duracion in [20,30,45,60]:
    for factor in [1.0,1.2,1.3]:
        base_costeable = (costeo_publico['no_show']*duracion/60*costeo_publico['haber_hora_contratada']*factor).sum()
        escenarios.append({
            'duracion_min':duracion,
            'factor_costo_empleador':factor,
            'capacidad_valorizada_profesiones_mapeadas':base_costeable,
            'cobertura_no_show':cobertura_no_show_costeable,
        })
sensibilidad_costos = pd.DataFrame(escenarios)
display(sensibilidad_costos.style.format({
    'factor_costo_empleador':'{:.1f}x',
    'capacidad_valorizada_profesiones_mapeadas':'${:,.0f}',
    'cobertura_no_show':'{:.1%}',
}))

costo_central_total = float(sensibilidad_costos.query('duracion_min==30 and factor_costo_empleador==1.0')['capacidad_valorizada_profesiones_mapeadas'].iloc[0])
recuperacion = pd.DataFrame({'tasa_reasignacion':[0.25,0.50,0.75]})
recuperacion['capacidad_valorizada_aprovechada'] = recuperacion['tasa_reasignacion']*costo_central_total
recuperacion['interpretacion'] = 'Valor de capacidad utilizada; no ahorro de caja'
display(recuperacion.style.format({'tasa_reasignacion':'{:.0%}','capacidad_valorizada_aprovechada':'${:,.0f}'}))

display(Markdown(f"""### Lectura de la aproximación

- La dotación vincula **{int(nomina_dotacion['cargo_analitico'].notna().sum())} de {len(nomina_dotacion)}** registros de nómina ({cobertura_personas:.1%}) y cubre {cobertura_haberes:.1%} de los haberes de julio.
- El mapa profesión–instrumento permite valorizar {no_show_mapeado:,} de {no_show_total:,} no-shows ({cobertura_no_show_costeable:.1%}). Los restantes se excluyen; no se les imputa un promedio institucional.
- Con **30 minutos por cupo** y sólo haberes (factor 1,0), la capacidad salarial valorizada para enero–julio es **${costo_central_total:,.0f}** en las profesiones mapeadas.
- Esta cifra no es ahorro ni costo marginal evitable. Es una aproximación al uso alternativo de capacidad contratada y debe leerse junto con la matriz de sensibilidad.
"""))



### Gastos generales y factores de absorción

El valor hora basado sólo en haberes subestima los recursos que sostienen la atención. Para incorporar gastos generales sin confundirlos con costos evitables, se construyen factores de absorción usando el gasto devengado promedio mensual de enero–junio y la nómina de julio como referencia.

Se mantienen cuatro vistas: **haberes**, **personal total**, **personal más bienes y servicios**, y **gasto total de Salud**. La última distribuye todos los gastos del Área sobre las horas contratadas y debe interpretarse como costo contable absorbido, no como costo marginal ni ahorro por una inasistencia.


In [ ]:

gasto_total_ene_jun = float(balance.loc[balance['codigo']=='215-00-00-000-000-000','devengado_acum'].iloc[0])
gasto_personal_ene_jun = float(balance.loc[balance['codigo']=='215-21-00-000-000-000','devengado_acum'].iloc[0])
gasto_bienes_servicios_ene_jun = float(balance.loc[balance['codigo']=='215-22-00-000-000-000','devengado_acum'].iloc[0])
haberes_julio = float(nomina['total_haberes'].sum())

factores_absorcion = pd.DataFrame([
    {'nivel':'1. Haberes directos', 'gasto_mensual_referencia':haberes_julio, 'factor_sobre_haberes':1.0,
     'interpretacion':'Capacidad salarial; excluye cargas y soporte'},
    {'nivel':'2. Personal total devengado', 'gasto_mensual_referencia':gasto_personal_ene_jun/6,
     'factor_sobre_haberes':(gasto_personal_ene_jun/6)/haberes_julio,
     'interpretacion':'Proxy de costo de personal completo'},
    {'nivel':'3. Personal + bienes y servicios', 'gasto_mensual_referencia':(gasto_personal_ene_jun+gasto_bienes_servicios_ene_jun)/6,
     'factor_sobre_haberes':((gasto_personal_ene_jun+gasto_bienes_servicios_ene_jun)/6)/haberes_julio,
     'interpretacion':'Agrega operación y suministros; absorción contable'},
    {'nivel':'4. Gasto total de Salud', 'gasto_mensual_referencia':gasto_total_ene_jun/6,
     'factor_sobre_haberes':(gasto_total_ene_jun/6)/haberes_julio,
     'interpretacion':'Costo institucional plenamente absorbido'},
])
factores_absorcion['capacidad_valorizada_30_min'] = costo_central_total*factores_absorcion['factor_sobre_haberes']
factores_absorcion['valor_medio_no_show_mapeado'] = factores_absorcion['capacidad_valorizada_30_min']/no_show_mapeado
display(factores_absorcion.style.format({
    'gasto_mensual_referencia':'${:,.0f}', 'factor_sobre_haberes':'{:.2f}x',
    'capacidad_valorizada_30_min':'${:,.0f}', 'valor_medio_no_show_mapeado':'${:,.0f}'
}))

costo_central_personal_total = float(factores_absorcion.loc[factores_absorcion['nivel'].str.startswith('2.'),'capacidad_valorizada_30_min'].iloc[0])
costo_central_personal_bienes = float(factores_absorcion.loc[factores_absorcion['nivel'].str.startswith('3.'),'capacidad_valorizada_30_min'].iloc[0])
costo_central_gasto_total = float(factores_absorcion.loc[factores_absorcion['nivel'].str.startswith('4.'),'capacidad_valorizada_30_min'].iloc[0])
factor_gasto_total = float(factores_absorcion.loc[factores_absorcion['nivel'].str.startswith('4.'),'factor_sobre_haberes'].iloc[0])

# Sensibilidad completa: duración x nivel de absorción.
sensibilidad_overhead=[]
for duracion in [20,30,45,60]:
    for _, row in factores_absorcion.iterrows():
        sensibilidad_overhead.append({
            'duracion_min':duracion,
            'nivel':row['nivel'],
            'factor':row['factor_sobre_haberes'],
            'capacidad_valorizada':costo_central_total*(duracion/30)*row['factor_sobre_haberes'],
        })
sensibilidad_overhead = pd.DataFrame(sensibilidad_overhead)
display(sensibilidad_overhead.pivot(index='duracion_min',columns='nivel',values='capacidad_valorizada').style.format('${:,.0f}'))

recuperacion_overhead = pd.DataFrame({'tasa_reasignacion':[0.25,0.50,0.75]})
recuperacion_overhead['capacidad_absorbida_aprovechada'] = recuperacion_overhead['tasa_reasignacion']*costo_central_gasto_total
recuperacion_overhead['interpretacion'] = 'Uso potencial de capacidad institucional; no ahorro de caja'
display(recuperacion_overhead.style.format({'tasa_reasignacion':'{:.0%}','capacidad_absorbida_aprovechada':'${:,.0f}'}))

display(Markdown(f"""#### Resultado con gastos generales

- El gasto mensual promedio de Salud es **${gasto_total_ene_jun/6:,.0f}**, frente a **${haberes_julio:,.0f}** de haberes de julio: factor de absorción **{factor_gasto_total:.2f}x**.
- En el escenario de 30 minutos, la valorización pasa de **${costo_central_total:,.0f}** en haberes a **${costo_central_personal_total:,.0f}** con personal total, **${costo_central_personal_bienes:,.0f}** con bienes y servicios, y **${costo_central_gasto_total:,.0f}** con el gasto total de Salud.
- La última cifra es costo contable absorbido. No demuestra que ese monto se pierda o pueda ahorrarse cuando una persona no asiste.
"""))



## Pérdida económica por capacidad clínica financiada no utilizada

La interpretación operacional se actualiza: el no-show no se presenta como ahorro potencial, sino como **pérdida económica por capacidad clínica financiada que no produjo la prestación esperada**. El gasto en personal, infraestructura y soporte ya fue realizado y la hora clínica no puede almacenarse.

Esta pérdida es distinta de una pérdida contable adicional: se trata del valor de recursos ya financiados cuyo producto sanitario no se materializó. La estimación supone que el cupo no fue reasignado y que existía demanda compatible. La fracción que una intervención logra reasignar se denomina **pérdida evitable**; el resto continúa como pérdida no recuperada.

El escenario central utiliza **45 minutos por cita**, **70% de jornada clínica efectiva** y **gasto total de Salud absorbido**. La proporción clínica corrige el supuesto anterior de que toda hora contratada estaba disponible para atención directa.


In [ ]:

DURACION_CENTRAL_PERDIDA_MIN = 45
PROPORCION_CLINICA_CENTRAL = 0.70

# El costo absorbido de v0.6 supone 30 minutos y 100% de jornada clínica.
perdida_central = costo_central_gasto_total * (DURACION_CENTRAL_PERDIDA_MIN/30) / PROPORCION_CLINICA_CENTRAL
perdida_central_mensual = perdida_central / 7
perdida_central_por_no_show = perdida_central / no_show_mapeado

sensibilidad_perdida=[]
for duracion in [30,45,60]:
    for proporcion_clinica in [0.60,0.70,0.80]:
        perdida = costo_central_gasto_total*(duracion/30)/proporcion_clinica
        sensibilidad_perdida.append({
            'duracion_min':duracion,
            'proporcion_jornada_clinica':proporcion_clinica,
            'perdida_ene_jul':perdida,
            'promedio_mensual':perdida/7,
            'perdida_por_no_show_mapeado':perdida/no_show_mapeado,
        })
sensibilidad_perdida = pd.DataFrame(sensibilidad_perdida)
display(sensibilidad_perdida.style.format({
    'proporcion_jornada_clinica':'{:.0%}', 'perdida_ene_jul':'${:,.0f}',
    'promedio_mensual':'${:,.0f}', 'perdida_por_no_show_mapeado':'${:,.0f}'
}))

matriz_perdida = sensibilidad_perdida.pivot(index='duracion_min',columns='proporcion_jornada_clinica',values='perdida_ene_jul')
matriz_perdida.columns = [f'{c:.0%} jornada clínica' for c in matriz_perdida.columns]
display(matriz_perdida.style.format('${:,.0f}'))

# Pérdida evitable bajo tasas de reasignación; no se denomina ahorro.
escenarios_perdida = pd.DataFrame({'tasa_reasignacion_efectiva':[0.25,0.50,0.75]})
escenarios_perdida['perdida_evitable'] = perdida_central*escenarios_perdida['tasa_reasignacion_efectiva']
escenarios_perdida['perdida_residual'] = perdida_central-escenarios_perdida['perdida_evitable']
escenarios_perdida['condicion'] = 'Requiere demanda compatible y reasignación observada'
display(escenarios_perdida.style.format({
    'tasa_reasignacion_efectiva':'{:.0%}', 'perdida_evitable':'${:,.0f}', 'perdida_residual':'${:,.0f}'
}))

display(Markdown(f"""### Resultado central actualizado

- **Pérdida económica estimada enero–julio:** ${perdida_central:,.0f}.
- **Promedio mensual:** ${perdida_central_mensual:,.0f}.
- **Pérdida media por no-show mapeado:** ${perdida_central_por_no_show:,.0f}.
- Supuestos: {DURACION_CENTRAL_PERDIDA_MIN} minutos, {PROPORCION_CLINICA_CENTRAL:.0%} de jornada clínica, gasto total absorbido y {cobertura_no_show_costeable:.1%} de cobertura de no-shows.
- El monto representa producto sanitario financiado que no se materializó. No implica que el mismo monto pueda recuperarse en efectivo.
"""))


## Puerta financiera y límites de la estimación de pérdida

### Resultado actualizado de la puerta financiera

La dotación de agosto permite asociar cargo y jornada a 316 de 327 registros de nómina. Con ello se habilita una **primera aproximación**, no un costeo definitivo:

- la profesión se obtiene con alta cobertura, pero nómina y dotación corresponden a meses distintos;
- el puente instrumento Rayen–cargo se limita a equivalencias semánticas explícitas y cubre 90,5% de los no-shows;
- la duración del cupo se trata como supuesto de sensibilidad (20, 30, 45 o 60 minutos);
- los haberes no incluyen aportes patronales ni equivalen al costo total empleador;
- la hora contratada no distingue actividad clínica de tareas administrativas, permisos o capacitación;
- sin demanda y reasignación observada, la capacidad valorizada no constituye beneficio realizado.

El escenario v0.7 usa 45 minutos, 70% de jornada clínica y gasto total absorbido. Las matrices de sensibilidad mantienen visibles los supuestos y evitan presentar una cifra única como verdad observada.

In [ ]:
puerta_financiera = pd.DataFrame([
    {'requisito':'Nómina agregada y reconciliada','estado':'PASS','impacto':'Permite haber por hora contratada'},
    {'requisito':'Cargo/profesión','estado':'PASS CON CAVEAT','impacto':f'{cobertura_personas:.1%} de personas; julio versus agosto'},
    {'requisito':'Unidad/establecimiento clínico','estado':'PARCIAL','impacto':'Los códigos requieren diccionario y mayor granularidad'},
    {'requisito':'Puente instrumento–cargo','estado':'PARCIAL','impacto':f'Equivalencias explícitas; cubre {cobertura_no_show_costeable:.1%} de no-shows'},
    {'requisito':'Aportes patronales','estado':'BLOQUEADO','impacto':'Se presenta sensibilidad 1,0x–1,3x; no costo total empleador'},
    {'requisito':'Horas clínicas disponibles','estado':'BLOQUEADO','impacto':'Hora contratada no equivale a hora clínica'},
    {'requisito':'Duración programada del cupo','estado':'SUPUESTO','impacto':'Sensibilidad de 20, 30, 45 y 60 minutos'},
    {'requisito':'Demanda y reasignación observada','estado':'BLOQUEADO','impacto':'No hay beneficio recuperable validado'},
])
display(puerta_financiera)

## Decisión sobre modelamiento predictivo

No se habilita aún un modelo predictivo. Antes se requiere:

1. una cita programada elegible con identificador robusto;
2. cancelaciones y estados finales comparables;
3. variables disponibles antes de la cita;
4. partición temporal y controles de fuga;
5. una intervención operacional definida;
6. evaluación de calibración, costo y equidad.

La prioridad inmediata es validar la línea base y el proceso de liberación/reasignación, no maximizar AUC.

## Hallazgos, límites y próximos pasos

Las cifras ejecutadas se resumen en la siguiente celda. La recomendación es provisional y depende de resolver los bloqueos señalados.

In [ ]:
promedio_central_por_no_show_mapeado = costo_central_total/no_show_mapeado
min_sensibilidad = sensibilidad_costos['capacidad_valorizada_profesiones_mapeadas'].min()
max_sensibilidad = sensibilidad_costos['capacidad_valorizada_profesiones_mapeadas'].max()

resumen_final = pd.DataFrame([
    {'dimension':'Línea base', 'resultado':f'{tasa_candidata:.2%} ({n_no_show:,}/{n_total:,})', 'estatus':'CANDIDATA; no institucional'},
    {'dimension':'Ambigüedad Rayen', 'resultado':f'{len(solapadas):,} claves cerrada/no-show', 'estatus':'Excluir y validar'},
    {'dimension':'Cobertura Keirón', 'resultado':f'{cobertura_keiron:.1%}', 'estatus':'Asociativa; no causal'},
    {'dimension':'Balance', 'resultado':f"${balance.loc[balance['codigo']=='215-00-00-000-000-000','devengado_acum'].iloc[0]:,.0f}", 'estatus':'Reconciliado a junio'},
    {'dimension':'Nómina', 'resultado':f"${nomina['total_haberes'].sum():,.0f} en haberes; {nomina['jornada_semanal'].sum()/44:.2f} FTE", 'estatus':'Julio; no costo empleador'},
    {'dimension':'Dotación', 'resultado':f'{cobertura_personas:.1%} de nómina; {cobertura_haberes:.1%} de haberes', 'estatus':'Agosto; período distinto'},
    {'dimension':'Capacidad salarial central', 'resultado':f'${costo_central_total:,.0f}', 'estatus':'30 min, 1,0x; 90,5% de no-shows'},
    {'dimension':'Sensibilidad de capacidad', 'resultado':f'${min_sensibilidad:,.0f}–${max_sensibilidad:,.0f}', 'estatus':'20–60 min; factor 1,0x–1,3x'},
    {'dimension':'Costo absorbido total', 'resultado':f'${costo_central_gasto_total:,.0f}', 'estatus':f'Intermedio v0.6: 30 min; factor {factor_gasto_total:.2f}x'},
    {'dimension':'Pérdida económica central', 'resultado':f'${perdida_central:,.0f}', 'estatus':'45 min; 70% jornada clínica; enero–julio'},
    {'dimension':'Pérdida mensual promedio', 'resultado':f'${perdida_central_mensual:,.0f}', 'estatus':'Promedio simple de 7 meses'},
    {'dimension':'Modelo predictivo', 'resultado':'No habilitado', 'estatus':'Primero validar target y decisión'},
])
display(resumen_final)

display(Markdown(f"""### Conclusión ejecutiva

- La tasa calculada es **{tasa_candidata:.2%}**, con {n_no_show:,} no-shows sobre {n_total:,} citas candidatas; no debe comunicarse aún como tasa institucional.
- Keirón se vinculó con **{cobertura_keiron:.1%}** de las citas candidatas; las diferencias por estado son asociaciones, no efectos del recordatorio.
- La dotación vincula **{cobertura_personas:.1%}** de la nómina y permite valorizar **{cobertura_no_show_costeable:.1%}** de los no-shows mediante profesiones con equivalencia explícita.
- En el escenario central de 30 minutos y sólo haberes, la capacidad salarial asociada asciende a **${costo_central_total:,.0f}**, aproximadamente **${promedio_central_por_no_show_mapeado:,.0f} por no-show mapeado**. El rango de sensibilidad es ${min_sensibilidad:,.0f}–${max_sensibilidad:,.0f}.
- Al corregir por 45 minutos y 70% de jornada clínica, la **pérdida económica estimada** asciende a **${perdida_central:,.0f} entre enero y julio**, o **${perdida_central_mensual:,.0f} mensuales**.
- Es una pérdida operacional por producto sanitario financiado que no se materializó. La pérdida evitable sólo puede comprobarse mediante demanda y reasignación observadas; no equivale a ahorro de caja ni demuestra impacto causal. Se recomienda validar duración, horas clínicas, costo empleador, unidades y reasignación antes de usar el monto para una decisión presupuestaria.
"""))